# Volume 7 Drills — Exploration & Meta-Learning

Work through each drill problem. Code cells are provided for problems 5 and 10.

## Drill 1 — Hard Exploration Problems

**Question:** What makes some environments **hard to explore**? Give two characteristics.

**Answer (fill in):** ___

<details><summary>Answer</summary>

Hard exploration environments typically have:

1. **Sparse rewards:** The agent receives a reward signal only rarely (e.g., only upon completing the entire task). Random exploration almost never finds the reward, so the agent has nothing to learn from.
   *Examples: Montezuma's Revenge, maze navigation, sparse-reward robotics.*

2. **Large state space:** The number of states is astronomically large, so random walks cannot cover it in reasonable time.

3. **Long horizons:** The reward requires executing a long sequence of correct actions — the probability of reaching it by chance is exponentially small.

4. **Deceptive local optima:** There may be small rewards that trap the agent in suboptimal behaviors, preventing discovery of the globally rewarding strategy.

Standard ε-greedy and Gaussian noise exploration fail on hard problems; dedicated methods (intrinsic motivation, go-explore, curiosity) are needed.
</details>

## Drill 2 — UCB1: Action Selection Formula

**Question:** Derive the UCB1 action selection formula for multi-armed bandits. Define each term.

**Answer (fill in):** ___

<details><summary>Answer</summary>

$$a^* = \arg\max_a \left[ \hat{\mu}_a + c \sqrt{\frac{\ln t}{N_a}} \right]$$

Where:
- **$\hat{\mu}_a$** — empirical mean reward of arm a (exploitation term)
- **$c$** — exploration constant (often $\sqrt{2}$ for theoretical guarantees)
- **$t$** — total number of pulls so far (timestep)
- **$N_a$** — number of times arm a has been pulled (exploration term)

The $\sqrt{\ln t / N_a}$ term is an **upper confidence bound** on the true mean. Arms pulled rarely have a large UCB (high uncertainty → high optimism). This implements **optimism in the face of uncertainty**: always act as if the unknown arm might be the best.

UCB1 achieves **O(log T) regret** — nearly optimal.
</details>

## Drill 3 — Intrinsic Motivation: Curiosity-Based Reward

**Question:** In curiosity-driven exploration, what is the **intrinsic reward** signal?

**Answer (fill in):** ___

<details><summary>Answer</summary>

The intrinsic reward in **ICM (Intrinsic Curiosity Module)** is the **prediction error** of a forward model:

$$r^{int}_t = \left\| \hat{\phi}(s_{t+1}) - \phi(s_{t+1}) \right\|^2$$

Where φ(s) is a learned feature embedding and $\hat{\phi}(s_{t+1})$ is the predicted next feature state.

High intrinsic reward = the model **failed to predict** what happened = the transition was **surprising/novel**. The agent is incentivized to visit states where the model is uncertain, naturally driving exploration into unvisited regions.

The total reward is: `r_total = r_extrinsic + β * r_intrinsic`
</details>

## Drill 4 — RND: Random Network Distillation

**Question:** How does **Random Network Distillation (RND)** generate intrinsic rewards?

**Answer (fill in):** ___

<details><summary>Answer</summary>

RND uses two networks:

1. **Target network** f(s): A **fixed, randomly initialized** network that maps states to feature vectors. It never changes.

2. **Predictor network** f̂(s): Trained to predict the output of the target network.

**Intrinsic reward** = prediction error:
$$r^{int}(s) = \left\| f̂(s) - f(s) \right\|^2$$

**Key insight:** The predictor gets better at predicting f(s) for **frequently visited states** (many training examples). For **novel states**, the predictor has poor accuracy → high prediction error → high intrinsic reward.

Advantages over ICM:
- Simpler (no inverse dynamics model)
- No "noisy TV" problem (doesn't get distracted by stochastic transitions)
- Achieved SOTA on Montezuma's Revenge
</details>

## Drill 5 — Code: UCB1 Action Selection

In [ ]:
import numpy as np

class UCB1Bandit:
    """
    Multi-armed bandit solved with UCB1 action selection.
    """

    def __init__(self, n_arms, c=np.sqrt(2)):
        self.n_arms = n_arms
        self.c = c
        self.counts = np.zeros(n_arms)     # N_a: number of pulls
        self.values = np.zeros(n_arms)     # mu_hat_a: empirical means
        self.t = 0                         # total steps

    def select_action(self):
        """Select arm with highest UCB score."""
        # Pull each arm at least once first (initialization)
        if self.t < self.n_arms:
            return self.t
        # UCB1 formula
        ucb_scores = self.values + self.c * np.sqrt(np.log(self.t) / self.counts)
        return int(np.argmax(ucb_scores))

    def update(self, arm, reward):
        """Update empirical mean using incremental formula."""
        self.t += 1
        self.counts[arm] += 1
        # Incremental update: mu = mu + (r - mu) / n
        self.values[arm] += (reward - self.values[arm]) / self.counts[arm]

# Test on a 3-arm bandit with known means
np.random.seed(42)
true_means = [0.3, 0.7, 0.5]  # arm 1 is best
bandit = UCB1Bandit(n_arms=3)

n_steps = 500
choices = []
for _ in range(n_steps):
    arm = bandit.select_action()
    reward = np.random.binomial(1, true_means[arm])
    bandit.update(arm, reward)
    choices.append(arm)

print(f"True means:      {true_means}")
print(f"Empirical means: {bandit.values.round(4)}")
print(f"Pull counts:     {bandit.counts.astype(int)}")
print(f"Best arm pulls:  {(np.array(choices) == 1).sum()} / {n_steps} ({(np.array(choices)==1).mean()*100:.1f}%)")

## Drill 6 — Go-Explore: Two Phases

**Question:** The Go-Explore exploration algorithm operates in two phases. What are they?

**Answer (fill in):** ___

<details><summary>Answer</summary>

1. **Go phase (Return to frontier):** Select a promising state from an **archive** of previously discovered states (choosing states that are rare or poorly explored). **Restore** the environment to that state (requires deterministic simulator or save states).

2. **Explore phase (Explore from there):** From the restored frontier state, take **random actions** to discover new states. Add all newly discovered states to the archive.

**Key insight:** Classic exploration suffers from "detachment" — the agent discovers a new area but can't reliably return to it due to stochasticity. Go-Explore decouples exploration from the navigation problem by **teleporting** back to frontier states.

After the exploration phase, a policy is trained via **imitation learning** on the found trajectories ("robustification" phase).

Go-Explore achieved superhuman performance on Montezuma's Revenge.
</details>

## Drill 7 — MAML: What Does It Optimize For?

**Question:** What does **MAML (Model-Agnostic Meta-Learning)** optimize for?

**Answer (fill in):** ___

<details><summary>Answer</summary>

MAML optimizes for a **meta-objective**: find initial parameters θ such that after **one or a few gradient steps** on a new task, the model performs well on that task.

$$\theta^* = \arg\min_\theta \sum_{\mathcal{T}_i \sim p(\mathcal{T})} \mathcal{L}_{\mathcal{T}_i}(\theta - \alpha \nabla_\theta \mathcal{L}_{\mathcal{T}_i}(\theta))$$

The outer loop finds θ; the inner loop simulates one gradient step on a task.

**In RL:** MAML finds a policy initialization that can quickly adapt to new environments with k rollouts. The meta-gradient requires backpropagating **through** the gradient step (second-order gradients), which is expensive. FOMAML and Reptile use first-order approximations.
</details>

## Drill 8 — RL²: Meta-Learning with an RNN

**Question:** How does **RL²** use a recurrent neural network (RNN) for meta-learning?

**Answer (fill in):** ___

<details><summary>Answer</summary>

RL² treats the entire **RL algorithm as a learned recurrent policy**:

1. The agent is an **RNN** (LSTM/GRU) that receives `(o_t, a_{t-1}, r_{t-1}, done_{t-1})` as input.

2. The **hidden state** of the RNN serves as the agent's memory — it implicitly encodes the task identity, past observations, and anything useful about the MDP.

3. The RNN is trained with standard RL across **many different tasks** (sampled from a task distribution). The **hidden state is never reset within a trial** but is reset between trials.

4. At meta-test time, the RNN's hidden state **accumulates information** from the current episode to perform in-context adaptation — the learned recurrence implements the exploration-exploitation strategy.

**Key insight:** The meta-learning is implicit — the RNN's recurrence learns to be its own RL algorithm.
</details>

## Drill 9 — Count-Based Exploration

**Question:** In count-based exploration, how are visit counts N(s,a) used to encourage exploration?

**Answer (fill in):** ___

<details><summary>Answer</summary>

Count-based exploration adds an **intrinsic bonus** inversely proportional to visit frequency:

$$r^{int}(s, a) = \frac{\beta}{\sqrt{N(s, a)}}$$

Or using pseudo-counts from a density model: $r^{int} \propto 1/\sqrt{\hat{N}(s)}$

**Update rule:** After taking action a in state s:
- N(s, a) ← N(s, a) + 1
- Augmented reward: r_total = r_extrinsic + β / √N(s, a)

**Effect:** Under-visited (s,a) pairs have large intrinsic bonuses → agent is incentivized to explore them. As visits increase, the bonus decays → agent eventually focuses on exploitation.

**Limitation:** Exact counts don't scale to large/continuous state spaces. Pseudo-count methods (e.g., using generative models) extend this to high-dimensional observations.
</details>

## Drill 10 — Code: Epsilon Decay with Exploration Bonus

In [ ]:
import numpy as np

def epsilon_decay(step, eps_start=1.0, eps_end=0.05, decay_steps=50000):
    """Exponential epsilon decay."""
    decay_rate = np.log(eps_end / eps_start) / decay_steps
    return eps_start * np.exp(decay_rate * min(step, decay_steps))

def count_bonus(visit_count, beta=1.0):
    """Intrinsic bonus from visit count: β / sqrt(N)."""
    return beta / np.sqrt(max(visit_count, 1))

def augmented_reward(extrinsic_r, visit_count, beta=0.1):
    """Total reward with exploration bonus."""
    return extrinsic_r + count_bonus(visit_count, beta)

# Show epsilon schedule
print("Epsilon decay schedule:")
for step in [0, 5000, 10000, 25000, 50000, 100000]:
    eps = epsilon_decay(step)
    print(f"  step={step:>7}: ε={eps:.4f}")

print()
print("Count-based exploration bonus:")
for n in [1, 4, 9, 16, 25, 100]:
    bonus = count_bonus(n, beta=1.0)
    print(f"  N={n:>4}: bonus={bonus:.4f}")

print()
print("Augmented rewards (r_ext=0.5, β=0.1):")
for n in [1, 10, 100, 1000]:
    r_total = augmented_reward(0.5, n, beta=0.1)
    print(f"  N={n:>5}: r_total={r_total:.4f}")